# Time Series Tools: ACF PACF

In this notebook you will rebuild the ACF(auto correlation function) and the PACF (Partial Autocorrelation Function) to investigate the class of ARMA models. 


## Load packages

In [ ]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.stattools import adfuller
from scipy import signal
import seaborn as sns
%matplotlib inline

# If you want a style choose one
#plt.style.use('Solarize_Light2')
#plt.style.use('tableau-colorblind10')
NF_ORANGE = '#ff5a36'
NF_BLUE = '#163251'

See all matplotlib styles under [matplotlib styles](https://problemsolvingwithpython.com/06-Plotting-with-Matplotlib/06.13-Plot-Styles/)

## Define auxiliary functions

The following functions are defined to simplify time series plotting, time differencing and stationarity testing. We will use them through this notebook.

In [ ]:
def plot_ts(ts = None, ts_add = None, title ='Time Series', legend=['1']):
    """
    Plots one or two time series in a single plot
    
        Args:
        ts: 1d- or 2d-array of time series. Dimension
            must be (n,) or (n,2)
        title: Title for the time plot.
        legend: list of legend names. If empty no legend.
        
        Returns:
        matplotlib plot object
    """
    plt.figure(figsize=(16, 6))
    plt.plot(ts[:,], color=NF_ORANGE)
    plt.grid(True,axis='y')
    plt.title(title)
    if ts_add is not None:
        plt.plot(ts_add, color=NF_BLUE)
    if len(legend) > 0:
        plt.legend(legend)
    
    plt.savefig("temp.png",dpi=200)
    plt.show()

def plot_acf_pacf(ts, lags=10, layout='h'):
    """
    Plots the empirical ACF and PACF of a time series process.
    
        Args:
        ts: array of time series
        
        Returns:
        matplotlib subplot with ACF and PACF
    """
    if layout == 'h':
        fig, ax = plt.subplots(1, 2, figsize = (16,6))
    else: 
        fig, ax = plt.subplots(2, 1, figsize = (16,6))
    sm.tsa.graphics.plot_acf(ts,color = NF_ORANGE,lags=lags, ax = ax[0],alpha=0.05)
    sm.tsa.graphics.plot_pacf(ts,color = NF_ORANGE, lags = lags, ax = ax[1],alpha=0.05)    
    
    plt.savefig("temp.png",dpi=200)


def diff_series(ts, interval=1):
    """
    Differences a time series by a certain lag.
    
        Args:
        ts: array of 1d time series
        
        Returns:
        Differenced time series
    """
    diff = ts[interval:] - ts[:-interval]
    return diff

def kpss_test(ts):
    """
    Performs a KPSS test for the null hypothesis of stationarity.
    
        Args:
        ts: 1d time series
        
        Returns:
        Summary of test statistic and critical values
    """
    print ('Results of KPSS Test:')
    kpsstest = kpss(ts, regression='c', nlags='legacy')
    kpss_output = pd.Series(kpsstest[0:3], index=['Test Statistic','p-value','Lags Used'])
    for key,value in kpsstest[3].items():
        kpss_output['Critical Value (%s)'%key] = value
    print (kpss_output)

def adf_test(ts):
    """
    Performs a Dickey-Fuller test for the null hypothesis of
    non-stationarity.
    
        Args:
        ts: 1-d time series
    
        Returns:
        Printed test statistic and critical values.
    """
    print ('Results of Dickey-Fuller Test:')
    dftest = adfuller(ts, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used',
                                             'Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)

## Empirical ACF and PACF of an ARMA process
If we want to determine the number of lags to include into an AR/MA or ARMA model we can take advantage of the autocorrelation function (ACF) and partial autocorrelation function (PACF). Both are function of the lag $k$ between two variables in time $X_t$ and $X_{t-k}$. So a observation $k$ days earlier or $k$ days later. The question is how much such observations influence each other either directly (ACF) or directly and indirectly (PACF). 

#### Empirical ACF and PACF for AR and MA process
In the following we take a look at simulated ARMA processes and their ACF and PACF functions.

In [ ]:
# Sample an AR(3) process and plot the ACF and PACF
np.random.seed(42)
process=sm.tsa.ArmaProcess(ar=[1, -0.4, 0.7, -0.1])
#process=sm.tsa.ArmaProcess(ma=[1,0.7])
#process=sm.tsa.ArmaProcess(ar=[1,0.9,0.8,0.7],ma=[1,0.7])
y = process.generate_sample(nsample=100)


In [ ]:
plot_ts(y, legend=['AR(3)-Process'], title='')
plt.savefig("AR(3)-TS.png",dpi=200)

Lets check for stationarity because thisis always good practise when dealing with timeseries.

In [ ]:
print('The process is stanionary: ',process.isstationary)

In [ ]:
adf_test(y)

In [ ]:
kpss_test(y)

Both tests suggest stationarity (ADF lets us reject the Nullhypothesis of it beeing non stationary,kpps says the nullhypothesis of stationarity holds).

Next lets turn our signal (i.e. our timeseries) into a DataFrame for the sake of easy manipulation. Additionally we add columns for the first 3 Lags.

In [ ]:
df=pd.DataFrame()
df=df.assign(y=y)
df=df.assign(y=y,
          yL1=df.y.shift(1),
          yL2=df.y.shift(2),
          yL3=df.y.shift(3))


In [ ]:
df.head()

As we can see, there are now NaNs in our Dataframe, because the first observations don't have lagged values. Hence, we just drop the first observations.

In [ ]:
df=df.dropna()

### Statsmodel PACF / ACF plots

Lets start by using the build-in functions from statsmodel. Afterwards we will develop this on our own:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (12,4))
    
sm.tsa.graphics.plot_acf(df.y,color = NF_ORANGE,lags=10, ax = ax[0],alpha=0.05)
sm.tsa.graphics.plot_pacf(df.y,color = NF_ORANGE, lags = 10, ax = ax[1],alpha=0.05)    
plt.savefig("ACF_PACF_signal.png",dpi=200)

### ACF

With some experience these plots give us an indication, that the process could be an AR(2) or AR(3) process (PACF cuts off after 2/3 Lags, ACF is oscillating). But thats not the point here.
Lets start with the ACF, the autocorrelation plot: this plot is supposed to show us, the correlation between the signal and lagged values of the signal. We can directly get the lagged values of the signal by looking into our Dataframe: During its creation we have already added the first three lagged values as features to it (yL1 for the first lagged one, yL2 for the second etc.)
From this Dataframe we can very conveniently run df.corr() to directly get to the desired correlation coefficients.

In [ ]:
print("data:")
display(df.head(10))
print("correlations:")
display(df.corr()["y"])

These correlation coefficients are (almost) exactly the values that are shown in the ACF plot. Some deviations betwen 'our' values and the once in the graphic plot are due to numerical issues (rounding etc), different algorithms to compute the correlation and slightly varying definitions of the coefficients. But let's not worry too much about the small differences here.
Of cause we can also get the numerical values that are used to generate the graphical plot:

In [ ]:
print("correlations:")
pd.Series(sm.tsa.acf(df.y,nlags=3,fft=False))

## PACF
The ACF was pretty staright-forward right? The PACF, unfortunatly is not-- except  for the first value which is exactly the same as in the ACF. However, instead of computing the correlation using df.corr(), we will use a linear regression (OLS) to evaluate the relation between the signal and the first lagged value:

In [ ]:
y=df.y
#X= sm.add_constant(df.yL1) #if you want you can try out whether including the intercept makes sense, however in our synthetic example it is irrelevant.

X=df.yL1
PACF_OLS_1=sm.OLS(y,X) #Predict the signal from 1. lagged value

RES_PACF_OLS_1=PACF_OLS_1.fit()
display(RES_PACF_OLS_1.summary())

RESID_PACF_OLS_1=RES_PACF_OLS_1.resid #Here it gets interesing (see below())

df=df.assign(yL1_decorr=RES_PACF_OLS_1.resid)

As we can see from the summary we only get a small Rˆ2. However, the featuer yL1 is still significant! After having trained the model, we are saving the Residuals. This is the most important step and the difference between the PACF and the ACF:
The residual of the model is the part of the signal, that can not be sufficiently explained by the first corelation. You could say the residuals are what remains of the signal, after the explained variation from yL1 is removed.

To build all the values for the pacf, these steps are repeated: a linear model is fit to the remaining signal, and the next signal we look at is the residuals from the last fit.

This way the values from the pacf are:
1. corr(y,L(1)y)
2. corr(RESIDUALS of OLS(y,L(1)y),L(2)y )
2. corr(RESIDUALS of OLS(y,L(2)y),L(3)y )
etc

In [ ]:
y=df.yL1_decorr
X= sm.add_constant(df.yL2)
X= df.yL2
PACF_OLS_2=sm.OLS(y,X)

RES_PACF_OLS_2=PACF_OLS_2.fit() #Predict the first residuals from 2. lagged value
display(RES_PACF_OLS_2.summary())
RESID_PACF_OLS_2=RES_PACF_OLS_2.resid

df=df.assign(yL2_decorr=RES_PACF_OLS_2.resid)

In [ ]:
y=df.yL2_decorr
X= sm.add_constant(df.yL3)
X= df.yL3
PACF_OLS_3=sm.OLS(y,X)

RES_PACF_OLS_3=PACF_OLS_3.fit()#Predict the second residuals from 3. lagged value
display(RES_PACF_OLS_3.summary())
RESID_PACF_OLS_3=RES_PACF_OLS_3.resid

df=df.assign(yL3_decorr=RES_PACF_OLS_3.resid)

In [ ]:

SM_PACF=pd.Series(sm.tsa.pacf(df.y,method="ols",nlags=3,))
print("Statsmodel PACF")
display(SM_PACF)
#display(display(df.corr()))

print("Our PACF")
Our_PACF=pd.Series([df.corr()['y']['y'],
                   df.corr()['y']['yL1'],
                   df.corr()['yL1_decorr']['yL2'],
                   df.corr()['yL2_decorr']['yL3']])

display(Our_PACF)

These correlation coefficients are (almost) exactly the values that are shown in the PACF plot. Some deviations betwen 'our' values and the once in the graphic plot are due to numerical issues (rounding etc), different algorithms to compute the correlation and slightly varying definitions of the coefficients. But let's not worry too much about the small differences here.
Of cause we can also get the numerical values that are used to generate the graphical plot:

In [ ]:

fig, ax = plt.subplots(2,3,figsize = (24,12))

sns.regplot(x=df.yL1,y=df.y,ax=ax[0,0],line_kws={"color": NF_ORANGE})
sns.regplot(x=df.yL2,y=df.yL1_decorr,ax=ax[0,1],line_kws={"color": NF_ORANGE})
sns.regplot(x=df.yL3,y=df.yL2_decorr,ax=ax[0,2],line_kws={"color": NF_ORANGE})

sns.regplot(x=df.yL1,y=df.yL1_decorr,ax=ax[1,0],line_kws={"color": NF_ORANGE})
sns.regplot(x=df.yL2,y=df.yL2_decorr,ax=ax[1,1],line_kws={"color": NF_ORANGE})
sns.regplot(x=df.yL3,y=df.yL3_decorr,ax=ax[1,2],line_kws={"color": NF_ORANGE})


for i in range(2):
    for j in range(3):
        ax[i,j].grid()
fig.suptitle('Steps for creating the PACF', fontsize=24,weight="bold")

ax[0,0].set_title("First step: fitting the signal to first lagged signal")
ax[1,0].set_title("Second step: taking the residuals from first step")
ax[0,1].set_title("Third step: fitting the first residuals to the second lagged signal")
ax[1,1].set_title("Fourth step: taking the residuals from third step")
ax[0,2].set_title("Fifth step: fitting the second residuals to the third lagged signal")
ax[1,2].set_title("Sixth step: taking the residuals from fifth step")


L1=f"R=PACF(1)={round(df.corr()['y']['yL1'],3)}"
L2=f"R=PACF(2)={round(df.corr()['yL1_decorr']['yL2'],3)}"
L3=f"R=PACF(3)={round(df.corr()['yL2_decorr']['yL3'],3)}"


for idx,label in enumerate([L1,L2,L3]):
    ax[0,idx].text(0.99, 0.95, label,
            verticalalignment='top', horizontalalignment='right',
            transform=ax[0,idx].transAxes,
            color='black', fontsize=12,weight="bold")

fig.savefig("Setps_PACF.png",dpi=200)
                   
                   

Finally, we can play around and generate some ARMA process signals and investigate how the resulting PACF and ACF looks like. Feel free to change the process!

In [ ]:
# Sample an AR(3) process and plot the ACF and PACF
np.random.seed(42)
process=sm.tsa.ArmaProcess(ar=[1, 0.9], ma=[1, -0.2])
y = process.generate_sample(nsample=1000)
print("Is stationary: ",process.isstationary)
plot_ts(y, legend=['ARMA(1,1)-Process'], title='')
plot_acf_pacf(y)
